In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Singularity Machine Learning - Classification: una Qiskit Function di Multiverse Computing
*Consulta il [riferimento API](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity)*


<Accordion>
<AccordionItem title="Versioni dei pacchetti">

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Si raccomanda di usare queste versioni o versioni più recenti.

```
scikit-learn~=1.8.0
```
</AccordionItem>
</Accordion>
> **Note:** * Le Qiskit Functions sono una funzionalità sperimentale disponibile solo per gli utenti dei piani IBM Quantum&reg; Premium, Flex e On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.

## Panoramica
Con la funzione "Singularity Machine Learning - Classification", puoi risolvere problemi reali di machine learning su hardware quantistico senza necessitare di competenze specifiche in ambito quantistico. Questa funzione applicativa, basata su metodi ensemble, è un classificatore ibrido. Sfrutta metodi classici come boosting, bagging e stacking per l'addestramento iniziale dell'ensemble. Successivamente, algoritmi quantistici come il variational quantum eigensolver (VQE) e il quantum approximate optimization algorithm (QAOA) vengono impiegati per migliorare la diversità, le capacità di generalizzazione e la complessità complessiva dell'ensemble addestrato.

A differenza di altre soluzioni di machine learning quantistico, questa funzione è in grado di gestire dataset su larga scala con milioni di esempi e feature senza essere limitata dal numero di qubit nel QPU di destinazione. Il numero di qubit determina soltanto la dimensione dell'ensemble che può essere addestrato. È anche molto flessibile e può essere utilizzata per risolvere problemi di classificazione in un'ampia gamma di domini, tra cui finanza, sanità e cybersicurezza.
Raggiunge costantemente alte accuratezze su problemi classicamente difficili che coinvolgono dataset ad alta dimensionalità, rumorosi e sbilanciati.
![Come funziona](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
È pensata per:
1. Ingegneri e data scientist di aziende che desiderano potenziare la propria offerta tecnologica integrando il machine learning quantistico nei propri prodotti e servizi,
2. Ricercatori di laboratori di ricerca quantistica che esplorano applicazioni di machine learning quantistico e vogliono sfruttare il calcolo quantistico per attività di classificazione, e
3. Studenti e docenti di istituzioni educative in corsi come quello di machine learning, che desiderano dimostrare i vantaggi del calcolo quantistico.

Il seguente esempio ne mostra le varie funzionalità, tra cui `create`, `list`, `fit` e `predict`, e ne dimostra l'utilizzo su un problema sintetico composto da due semicerchi a forma di mezza luna che si intrecciano, un problema notoriamente difficile a causa del suo confine decisionale non lineare.


## Descrizione della funzione
Questa Qiskit Function permette agli utenti di risolvere problemi di classificazione binaria utilizzando il classificatore ensemble potenziato quantisticamente di Singularity. Dietro le quinte, usa un approccio ibrido per addestrare classicamente un ensemble di classificatori sul dataset etichettato, e poi lo ottimizza per massimizzare la diversità e la generalizzazione usando il Quantum Approximate Optimization Algorithm (QAOA) sui QPU IBM&reg;. Tramite un'interfaccia intuitiva, gli utenti possono configurare un classificatore in base alle proprie esigenze, addestrarlo sul dataset desiderato e usarlo per fare previsioni su un dataset mai visto in precedenza.

Per risolvere un generico problema di classificazione:
1. Preelabora il dataset e suddividilo in set di addestramento e di test. Facoltativamente, puoi suddividere ulteriormente il set di addestramento in set di addestramento e di validazione. Questo può essere ottenuto usando [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Se il set di addestramento è sbilanciato, puoi ricampionarlo per bilanciare le classi usando [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Carica separatamente i set di addestramento, validazione e test nell'archiviazione della funzione usando il metodo `file_upload` del catalogo, passando il percorso corrispondente ogni volta.
4. Inizializza il classificatore quantistico usando l'azione `create` della funzione, che accetta iperparametri come il numero e i tipi di learner, la regolarizzazione (valore lambda) e le opzioni di ottimizzazione inclusi il numero di livelli, il tipo di ottimizzatore classico, il backend quantistico e così via.
5. Addestra il classificatore quantistico sul set di addestramento usando l'azione `fit` della funzione, passandogli il set di addestramento etichettato e il set di validazione se applicabile.
6. Esegui previsioni sul set di test precedentemente non visto usando l'azione `predict` della funzione.

---
## Per iniziare
Autenticati usando la tua [chiave API di IBM Quantum Platform](http://quantum.cloud.ibm.com/) e seleziona la Qiskit Function come segue:

In [6]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [7]:
# load function
singularity = catalog.load("multiverse/singularity")

Esegui i seguenti passaggi:

1) Crea il dataset sintetico usando la funzione [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) di [scikit-learn](https://scikit-learn.org/).
2) Carica il dataset sintetico generato nella [directory dati condivisa](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Crea il classificatore potenziato quantisticamente usando l'azione [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create).
4) Elenca i tuoi classificatori usando l'azione [`list`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#list).
5) Addestra il classificatore sui dati di addestramento usando l'azione [`fit`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#fit).
6) Usa il classificatore addestrato per effettuare previsioni sui dati di test con l'azione [`predict`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#predict).
7) Elimina il classificatore usando l'azione [`delete`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#delete).
8) Fai pulizia quando hai finito.
**Passo 1.** Importa i moduli necessari e genera il dataset sintetico, poi suddividilo in dataset di addestramento e di test.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


**Step 2.** Save the labeled training and test datasets on your local disk, and then upload them to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Passo 2.** Salva i dataset di addestramento e di test etichettati sul disco locale, quindi caricali nella [directory dati condivisa](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Passo 3.** Crea un classificatore potenziato quantisticamente usando l'azione [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create).

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


**Step 6.** Delete the quantum-enhanced classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

### `create_fit_predict` example

The following example demonstrates the [`create_fit_predict`](/docs/api/functions/multiverse-computing-singularity#create_fit_predict) action.

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


**Passo 4.** Addestra il classificatore potenziato quantisticamente usando l'azione [`fit`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#fit).